# Chapter 10
Use the `pennylane` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import time
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp


def all_bitstrings(n_bits):
    """Return binary rows in ascending order, most-significant bit first."""
    if not isinstance(n_bits, (int, np.integer)) or n_bits < 1:
        raise ValueError("n_bits must be a positive integer")
    values = np.arange(2 ** n_bits)
    return (values[:, None] >> np.arange(n_bits)[::-1]) & 1


def validate_probabilities(prob, expected_size=None):
    """Check a finite normalized vector; ordering is checked separately."""
    values = np.asarray(qml.math.toarray(prob), dtype=float)
    size = values.size
    if values.ndim != 1 or size < 2 or size & (size - 1):
        raise ValueError("probabilities need a power-of-two vector length")
    if expected_size is not None and size != expected_size:
        raise ValueError("unexpected probability-vector shape")
    if not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError("probabilities must be finite and nonnegative")
    if not np.isclose(values.sum(), 1.0, rtol=0, atol=1e-8):
        raise ValueError("probabilities must sum to one")
    return values


def hamming_kernel(bitstrings, sigma):
    """Return the Gaussian kernel on binary rows for positive sigma."""
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma must be finite and positive")
    distance = (bitstrings[:, None, :] != bitstrings[None, :, :]).sum(-1)
    return np.exp(-distance / (2 * sigma ** 2))


def mmd2(p, q, kernel):
    """Return the exact quadratic discrepancy for aligned probability vectors."""
    difference = p - q
    return difference @ kernel @ difference


bits = all_bitstrings(4)
labels = ["".join(map(str, row)) for row in bits]
population_prob = np.zeros(16)
# A weighted bars-and-stripes target with one rare mode, 1010.
population_prob[[0, 15, 3, 12, 5, 10]] = [.30, .24, .20, .15, .10, .01]
train_ids = np.random.default_rng(123).choice(16, 600, p=population_prob)
target_prob = np.bincount(train_ids, minlength=16) / len(train_ids)
validate_probabilities(target_prob, 16)
kernel = hamming_kernel(bits, sigma=1.0)
print("First bitstrings:", labels[:4])
print("Training probabilities:", target_prob)


In [ ]:
wires = (0, 1, 2, 3)
layers = 3
device = qml.device("default.qubit", wires=wires, shots=None)


def qcbm_gates(theta):
    """Queue Hadamards followed by RZ/RY layers and a directed CNOT ring."""
    for wire in wires:
        qml.Hadamard(wires=wire)
    for layer in theta:
        for wire in wires:
            qml.RZ(layer[wire, 0], wires=wire)
            qml.RY(layer[wire, 1], wires=wire)
        for wire in wires:
            qml.CNOT(wires=[wire, (wire + 1) % len(wires)])


@qml.qnode(device, interface="autograd", diff_method="backprop")
def circuit_probabilities(theta):
    """Return exact probabilities in the declared wire order: 0000 to 1111."""
    qcbm_gates(theta)
    return qml.probs(wires=wires)


@qml.qnode(device)
def ordering_probe():
    """Prepare 1000 to check a nonsymmetric basis outcome."""
    qml.PauliX(wires=0)
    return qml.probs(wires=wires)


assert np.argmax(ordering_probe()) == 8
print("Wire-0 flip:", labels[np.argmax(ordering_probe())])


In [ ]:
def train_qcbm(theta, target_prob, n_steps=200,
               learning_rate=0.08, sigma=1.0):
    """Fit the four-qubit QCBM; return probabilities, losses, and costs."""
    target = validate_probabilities(target_prob, 16)
    if not isinstance(n_steps, (int, np.integer)) or n_steps < 1:
        raise ValueError("n_steps must be a positive integer")
    if not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("learning_rate must be finite and positive")
    initial = np.asarray(theta, dtype=float)
    if initial.shape != (layers, 4, 2) or not np.isfinite(initial).all():
        raise ValueError("theta must be finite with shape (3, 4, 2)")
    K = hamming_kernel(bits, sigma)
    weights = pnp.array(initial, requires_grad=True)
    optimizer = qml.AdamOptimizer(stepsize=learning_rate)
    history, objective_calls = [], 0

    def objective(current):
        nonlocal objective_calls
        objective_calls += 1
        probability = circuit_probabilities(current)
        validate_probabilities(probability, len(target))
        # Keep the differentiable tensor in the loss, not its checked copy.
        return mmd2(target, probability, K)

    started = time.perf_counter()
    with qml.Tracker(device) as tracker:
        for _ in range(n_steps):
            weights, loss = optimizer.step_and_cost(objective, weights)
            history.append(float(loss))
        history.append(float(objective(weights)))
        probability = validate_probabilities(circuit_probabilities(weights), 16)
    return {
        "theta": weights, "prob": probability, "history": np.array(history),
        "objective_calls": objective_calls,
        "executions": tracker.totals["executions"],
        "seconds": time.perf_counter() - started,
    }


runs = {}
for seed in (7, 19, 31):
    initial = np.random.default_rng(seed).normal(0, 0.3, (layers, 4, 2))
    runs[seed] = train_qcbm(initial, target_prob)
    print(seed, "initial/final MMD2:", runs[seed]["history"][[0, -1]])
# Same seed and architecture, deliberately insufficient optimization budget.
short_run = train_qcbm(
    np.random.default_rng(7).normal(0, 0.3, (layers, 4, 2)), target_prob, n_steps=5
)
main_run = runs[7]  # Predeclared example, not the best seed after evaluation.
for label, p, q in zip(labels, population_prob, main_run["prob"]):
    print(label, "population/model:", round(p, 5), round(q, 5))


In [ ]:
def sample_qcbm(theta, shots, seed):
    """Sample the trained circuit and return counts in the established bit order."""
    sample_device = qml.device("default.qubit", wires=wires, seed=seed)

    @qml.set_shots(shots)
    @qml.qnode(sample_device, interface=None)
    def draw():
        qcbm_gates(np.asarray(theta))
        return qml.sample(wires=wires)

    samples = np.asarray(draw())
    ids = samples @ (2 ** np.arange(4)[::-1])
    return np.bincount(ids, minlength=16)


def shot_report(draw_counts, shots, repeats=20):
    """Return mean/SD TV, observed coverage, rare misses, first histogram, runtime."""
    started = time.perf_counter()
    distributions = np.array([
        draw_counts(shots, 1000 + repeat) / shots for repeat in range(repeats)
    ])
    tv = np.abs(distributions - population_prob).sum(1) / 2
    observed = (distributions[:, population_prob > 0] > 0).mean(1)
    return {
        "TV_mean": tv.mean(), "TV_sd": tv.std(ddof=1),
        "observed_coverage": observed.mean(),
        "rare_miss_fraction": np.mean(distributions[:, 10] == 0),
        "first_histogram": distributions[0],
        "seconds": time.perf_counter() - started,
    }


shot_budgets = (32, 256, 2048)
shot_results = {}
for shots in shot_budgets:
    report = shot_report(
        lambda count, seed: sample_qcbm(main_run["theta"], count, seed), shots
    )
    shot_results[shots] = report
    print(shots, {k: round(float(v), 5) for k, v in report.items()
                  if k not in ("first_histogram", "seconds")})
    print("First histogram:", np.round(report["first_histogram"], 4))
    print("Exact probability of missing 1010:",
          round((1 - main_run["prob"][10]) ** shots, 5))


In [ ]:
def moments(prob):
    """Return bit-one marginals and the covariance matrix for a distribution."""
    marginal = prob @ bits
    covariance = (bits.T * prob) @ bits - np.outer(marginal, marginal)
    return marginal, covariance


def distribution_metrics(prob):
    """Compare a full distribution with the known population, without filtering."""
    prob = validate_probabilities(prob, 16)
    marginal, covariance = moments(prob)
    ref_marginal, ref_covariance = moments(population_prob)
    modes = population_prob > 0
    return {
        "TV": np.abs(prob - population_prob).sum() / 2,
        "MMD2": float(mmd2(population_prob, prob, kernel)),
        "coverage": np.mean(prob[modes] >= 0.001),
        "invalid_mass": prob[~modes].sum(),
        "marginal_error": np.max(np.abs(marginal - ref_marginal)),
        "covariance_error": np.linalg.norm(covariance - ref_covariance),
    }


for name, run in list(runs.items()) + [("early stop", short_run)]:
    print(name, {k: round(float(v), 5)
                 for k, v in distribution_metrics(run["prob"]).items()})


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 16})
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for seed, run in runs.items():
    axes[0, 0].semilogy(run["history"], label=f"seed {seed}")
axes[0, 0].semilogy(short_run["history"], "k--", label="5-step control")
axes[0, 0].set(xlabel="Adam updates", ylabel="Training MMD squared")
axes[0, 0].legend()

x = np.arange(16)
for offset, prob, name in [(-.25, population_prob, "population"),
                           (0, target_prob, "training"),
                           (.25, main_run["prob"], "QCBM")]:
    axes[0, 1].bar(x + offset, prob, width=.25, label=name)
axes[0, 1].set(xticks=x, xticklabels=labels, ylabel="Probability")
axes[0, 1].tick_params(axis="x", labelrotation=90, labelsize=14)
axes[0, 1].legend(fontsize=14, ncol=3, loc="lower center",
                  bbox_to_anchor=(0.5, 1.02))

for prob, name in [(population_prob, "population"),
                   (main_run["prob"], "QCBM seed 7"),
                   (short_run["prob"], "5-step control")]:
    axes[1, 0].plot(range(4), moments(prob)[0], "o-", label=name)
axes[1, 0].set(xlabel="Wire / row-major pixel", ylabel="P(bit = 1)",
               xticks=range(4), ylim=(0, 1))
axes[1, 0].legend(fontsize=14)

error = moments(main_run["prob"])[1] - moments(population_prob)[1]
bound = max(float(np.abs(error).max()), 1e-8)
heatmap = axes[1, 1].imshow(error, cmap="RdBu_r", vmin=-bound, vmax=bound)
axes[1, 1].set(title="Covariance error",
               xlabel="Wire", ylabel="Wire", xticks=range(4), yticks=range(4))
fig.colorbar(heatmap, ax=axes[1, 1])
fig.savefig("chapter10_diagnostics.pdf")
plt.show()


In [ ]:
from scipy.optimize import minimize
from scipy.special import logsumexp

baseline_prob, baseline_cost = {}, {}
started = time.perf_counter()
baseline_prob["uniform"] = np.full(16, 1 / 16)
baseline_cost["uniform"] = (0, time.perf_counter() - started)
started = time.perf_counter()
baseline_prob["empirical"] = target_prob.copy()
baseline_cost["empirical"] = (0, time.perf_counter() - started)
started = time.perf_counter()
means = target_prob @ bits
baseline_prob["independent"] = np.prod(
    np.where(bits == 1, means, 1 - means), axis=1
)
baseline_cost["independent"] = (0, time.perf_counter() - started)

# Full pairwise log-linear model: 4 bit terms plus 6 pair terms.
started = time.perf_counter()
features = np.column_stack([bits] + [
    bits[:, i] * bits[:, j] for i in range(4) for j in range(i + 1, 4)
])
training_moments = target_prob @ features
regularization = 1e-4


def pairwise_loss(weights):
    """Return regularized negative log likelihood and its exact gradient."""
    logits = features @ weights
    log_partition = logsumexp(logits)
    probability = np.exp(logits - log_partition)
    loss = (log_partition - training_moments @ weights
            + 0.5 * regularization * (weights @ weights))
    gradient = probability @ features - training_moments + regularization * weights
    return loss, gradient


pairwise_fit = minimize(pairwise_loss, np.zeros(10), jac=True,
                        method="L-BFGS-B", options={"maxiter": 500,
                        "gtol": 1e-9, "ftol": 1e-12})
logits = features @ pairwise_fit.x
baseline_prob["pairwise"] = np.exp(logits - logsumexp(logits))
baseline_cost["pairwise"] = (pairwise_fit.nfev, time.perf_counter() - started)
print("Pairwise optimizer:", pairwise_fit.success, pairwise_fit.message)

baseline_shots = {}
for name, prob in baseline_prob.items():
    print(name, {k: round(float(v), 5)
                 for k, v in distribution_metrics(prob).items()})
    baseline_shots[name] = {}
    for shots in shot_budgets:
        report = shot_report(
            lambda count, seed: np.random.default_rng(seed).multinomial(count, prob),
            shots,
        )
        baseline_shots[name][shots] = report
        print(name, shots, "TV mean/SD:", report["TV_mean"], report["TV_sd"],
              "coverage:", report["observed_coverage"],
              "rare misses:", report["rare_miss_fraction"])


In [ ]:
resources = qml.specs(circuit_probabilities)(main_run["theta"]).resources
print("Circuit parameters/depth/CNOTs:", main_run["theta"].size,
      resources.depth, resources.gate_types["CNOT"])
for name, run in list(runs.items()) + [("early stop", short_run)]:
    print("Exact training", name, "objectives/executions/seconds:",
          run["objective_calls"], run["executions"], round(run["seconds"], 3))
for shots, report in shot_results.items():
    print("Finite-shot QCBM evaluation", shots, "repeats/total shots/seconds:",
          20, 20 * shots, round(report["seconds"], 3))
for name, (evaluations, seconds) in baseline_cost.items():
    print("Classical fit", name, "objectives/seconds:", evaluations, seconds)
    for shots, report in baseline_shots[name].items():
        print("Classical sampling", name, shots, "draws/seconds:",
              20 * shots, report["seconds"])
print("No finite-shot training, device-noise simulation, or hardware run.")
